# 13 — SRNet-v13 progressive curriculum (TRAIN/DEV only)

This notebook is a **detector-development** experiment. It does not score TEST.

Patch 11 showed that starting a randomly initialized SRNet at 0.012 bpp and descending to 0.009 bpp did not move development AUC away from chance. Patch 12 then measured TRAIN-only payload feasibility and selected 0.015 bpp under the pre-specified >=0.90 feasibility rule. Patch 13 uses a stronger **bootstrap-only** curriculum:

\[
0.050 ightarrow 0.030 ightarrow 0.015 ightarrow 0.009\ 	ext{net bpp}.
\]

The 0.050 and 0.030 payloads are used only to bootstrap the steganalyzer on TRAIN/DEV; they are **not** new manuscript operating points and do not modify the frozen RDH allocator or the primary 0.009-bpp endpoint.

The notebook intentionally ends after a development sanity gate. TEST scoring, if ever performed, must be implemented in a separate later patch.

In [ ]:
from pathlib import Path
import gc, hashlib, json, os, time, joblib, yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

CPU_THREADS=max(1,min(8,os.cpu_count() or 1))
torch.set_num_threads(CPU_THREADS)
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass
torch.backends.mkldnn.enabled=True

from rdhlab.io import read_gray
from rdhlab.blockcodec import analyze_blocks
from rdhlab.pipeline import run_frozen_image_precomputed
from rdhlab.detectors import detector_metrics, paired_detector_bootstrap
from rdhlab.freeze_protocol import sha256_file, stable_id_hash
from rdhlab.srnet_secondary_v11 import score_srnet, parameter_count
from rdhlab.srnet_curriculum_v13 import train_progressive_srnet_v13, save_state_dict

config=yaml.safe_load(Path('/workspace/config/experiment.yaml').read_text())
seed=int(config['project']['seed'])
fixed_fpr=float(config['detectors']['fixed_fpr'])
confidence=float(config['statistics']['confidence'])
n_boot=max(5000,int(config['statistics'].get('bootstrap_resamples',5000)))

manifest=pd.read_csv(config['dataset']['prepared_manifest'])
train=manifest[manifest.split=='train'].reset_index(drop=True)
assert len(train)==6000
assert train.source_id.astype(str).is_unique

allocator_path=Path('/workspace/config/frozen_allocator.json')
allocator=json.loads(allocator_path.read_text())
alpha=float(allocator['alpha'])
primary_bpp=float(allocator['teacher_payload_bpp'])
bs=int(allocator.get('block_size',config['dataset']['block_size']))
assert np.isclose(alpha,0.25)
assert np.isclose(primary_bpp,0.009)

risk_path=Path('/workspace/results/models/srm_teacher_local_risk.joblib')
if sha256_file(risk_path)!=allocator['provenance_sha256']['srm_teacher_local_risk_joblib']:
    raise RuntimeError('Frozen local-risk model hash mismatch.')

OUT=Path('/workspace/results/srnet_curriculum_v13')
CACHE=OUT/'cache'
CKPT=OUT/'checkpoints'
MODEL_DIR=Path('/workspace/results/models')
for p in (OUT,CACHE,CKPT,MODEL_DIR):
    p.mkdir(parents=True,exist_ok=True)

print('PyTorch:',torch.__version__)
print('CUDA:',torch.cuda.is_available())
print('CPU threads:',torch.get_num_threads())
print('Frozen alpha:',alpha)
print('Primary payload:',primary_bpp)
print('Block size:',bs)
print('TRAIN images:',len(train))

## 1. Audit Patch 12 before defining Patch 13

The v12 result is treated as TRAIN-only detector-development evidence. Patch 13 does not reinterpret it as a primary study result.

In [ ]:
v12_dir=Path('/workspace/results/srnet_payload_probe_v12')
v12_sel=json.loads((v12_dir/'srnet_v12_payload_selection.json').read_text())
v12_summary=pd.read_csv(v12_dir/'srnet_v12_payload_probe_summary.csv')

assert v12_sel['status']=='STRONG_PAYLOAD_SELECTED'
assert np.isclose(float(v12_sel['selected_payload_bpp']),0.015)
assert bool(v12_sel['test_split_scored']) is False
assert bool(v12_sel['srnet_trained']) is False
assert bool(v12_sel['allocator_retuned']) is False
assert np.isclose(float(v12_sel['selected_feasible_fraction']),0.924)
assert np.isclose(float(v12_sel['selected_exact_recovery_fraction_feasible']),1.0)

required_payloads=[0.012,0.015,0.020,0.030,0.050]
for p in required_payloads:
    if not np.any(np.isclose(v12_summary.target_net_bpp.astype(float),p)):
        raise RuntimeError(f'Patch-12 summary missing payload {p}')

display(v12_summary)
print(json.dumps(v12_sel,indent=2))

## 2. Frozen detector-development split and progressive schedule

The fit/development split is unchanged from v11:

- fit: TRAIN[0:5250]
- development: TRAIN[5250:6000]

The bootstrap payload 0.050 bpp is deliberately stronger than the v12 broad-feasibility selection. This change is allowed only because it is made on TRAIN-only diagnostics after v11 failed and before any SRNet TEST scoring.

In [ ]:
FIT_N=5250
DEV_N=750
srnet_fit=train.iloc[:FIT_N].reset_index(drop=True)
srnet_dev=train.iloc[FIT_N:FIT_N+DEV_N].reset_index(drop=True)
assert len(srnet_fit)==FIT_N and len(srnet_dev)==DEV_N
assert set(srnet_fit.source_id.astype(str)).isdisjoint(set(srnet_dev.source_id.astype(str)))

PAIR_BATCH=6
WEIGHT_DECAY=2e-4
SANITY_AUC_THRESHOLD=0.65

STAGES=[
    {
        'name':'bootstrap_0p050','payload_bpp':0.050,
        'min_epochs':3,'max_epochs':6,'lr':1e-3,'pairs_per_epoch':2200,
        'early_success_auc':0.75,
        'fail_check_epoch':4,'fail_below_auc':0.55,
    },
    {
        'name':'bridge_0p030','payload_bpp':0.030,
        'min_epochs':2,'max_epochs':5,'lr':1e-4,'pairs_per_epoch':3000,
        'early_success_auc':0.70,
    },
    {
        'name':'bridge_0p015','payload_bpp':0.015,
        'min_epochs':2,'max_epochs':5,'lr':1e-4,'pairs_per_epoch':4000,
        'early_success_auc':0.67,
    },
    {
        'name':'target_0p009','payload_bpp':0.009,
        'min_epochs':4,'max_epochs':8,'lr':1e-4,'pairs_per_epoch':4500,
        'early_success_auc':SANITY_AUC_THRESHOLD,
    },
]

protocol={
    'analysis_status':'POST_HOC_SECONDARY_ROBUSTNESS_DETECTOR_DEVELOPMENT_V13',
    'target_journal':'Signal Processing',
    'detector':'SRNet architecture',
    'role':'second independently trained neural steganalyzer',
    'teacher':'SRM-derived local-risk teacher',
    'primary_confirmatory_detector':'enhanced residual CNN',
    'allocator_alpha_frozen':alpha,
    'primary_payload_bpp':0.009,
    'v12_selected_bridge_payload_bpp':0.015,
    'bootstrap_payloads_bpp':[0.050,0.030],
    'curriculum_path_bpp':[0.050,0.030,0.015,0.009],
    'fit_n':FIT_N,'dev_n':DEV_N,
    'pair_batch_size':PAIR_BATCH,
    'images_per_training_batch':2*PAIR_BATCH,
    'weight_decay':WEIGHT_DECAY,
    'stages':STAGES,
    'final_sanity_auc_threshold':SANITY_AUC_THRESHOLD,
    'fixed_fpr':fixed_fpr,
    'bootstrap_resamples':n_boot,
    'test_scoring_in_this_patch':False,
    'no_allocator_or_primary_endpoint_retuning_permitted':True,
    'v12_selection_sha256':sha256_file(v12_dir/'srnet_v12_payload_selection.json'),
    'v12_summary_sha256':sha256_file(v12_dir/'srnet_v12_payload_probe_summary.csv'),
    'allocator_sha256':sha256_file(allocator_path),
    'risk_model_sha256':sha256_file(risk_path),
    'fit_ids_sha256':stable_id_hash(srnet_fit.source_id.astype(str).tolist()),
    'dev_ids_sha256':stable_id_hash(srnet_dev.source_id.astype(str).tolist()),
    'published_srnet_training_not_reproduced':True,
    'note':'0.050/0.030 are TRAIN-only bootstrap payloads introduced after v11 failed; 0.015 remains the v12 pre-specified broad-feasibility bridge.'
}
protocol_path=OUT/'srnet_v13_protocol_pretest.json'
protocol_path.write_text(json.dumps(protocol,indent=2),encoding='utf-8')
protocol_sha=sha256_file(protocol_path)
print('PRE-TEST PROTOCOL SHA256:',protocol_sha)
print(json.dumps(protocol,indent=2))

## 3. Build and cache aligned random-allocation pairs

Each payload uses only feasible cover/stego pairs. Cover and stego receive identical geometric augmentation during training. The cache is bound to the frame ID hash, payload, block size, and v13 protocol hash.

In [ ]:
def random_order_for_plans(plans,source_id,payload):
    bids=np.asarray([p.block_id for p in plans],dtype=int)
    digest=hashlib.sha256(f'{seed}|srnet-v13|{payload:.6f}|{source_id}'.encode()).digest()
    rng=np.random.default_rng(int.from_bytes(digest[:8],'little'))
    out=bids.copy(); rng.shuffle(out)
    return out

def cache_key(label,payload):
    return f'{label}_{payload:.3f}'.replace('.','p')

def load_or_build_pairs(frame,label,payload):
    key=cache_key(label,payload)
    cp=CACHE/f'{key}_covers.npy'
    sp=CACHE/f'{key}_stegos.npy'
    ip=CACHE/f'{key}_ids.csv'
    mp=CACHE/f'{key}_meta.json'
    expected_hash=stable_id_hash(frame.source_id.astype(str).tolist())
    if cp.exists() and sp.exists() and ip.exists() and mp.exists():
        meta=json.loads(mp.read_text())
        if (meta.get('source_ids_sha256')==expected_hash and
            np.isclose(float(meta.get('payload_bpp',-1)),float(payload)) and
            int(meta.get('block_size',-1))==bs and
            meta.get('protocol_sha256')==protocol_sha):
            covers=np.load(cp,mmap_mode='r')
            stegos=np.load(sp,mmap_mode='r')
            ids=pd.read_csv(ip).source_id.astype(str).tolist()
            if len(covers)==len(stegos)==len(ids):
                print('CACHE HIT:',key,'pairs=',len(ids))
                return covers,stegos,ids
        raise RuntimeError(f'Stale/incompatible cache exists for {key}; inspect before deletion.')

    covers=[]; stegos=[]; ids=[]; skipped=[]
    for j,row in frame.iterrows():
        sid=str(row.source_id)
        x=read_gray(row.path)
        plans=analyze_blocks(x,bs)
        order=random_order_for_plans(plans,sid,payload)
        rr=run_frozen_image_precomputed(
            x,sid,float(payload),'random',{'random':order},[],bs,seed,False,None,plans=plans
        )
        if rr['feasible']:
            if not (rr['exact_image'] and rr['exact_message'] and float(rr['ber'])==0.0):
                raise RuntimeError(f'Reversibility invariant failed: {label} {payload} {sid}')
            covers.append(np.asarray(x,dtype=np.uint8))
            stegos.append(np.asarray(rr['stego'],dtype=np.uint8))
            ids.append(sid)
        else:
            skipped.append(sid)
        if (j+1)%250==0 or j+1==len(frame):
            print(label,payload,j+1,'/',len(frame),'feasible',len(covers),flush=True)

    if not covers:
        raise RuntimeError(f'No feasible pairs for {label} payload={payload}')
    covers=np.stack(covers)
    stegos=np.stack(stegos)
    np.save(cp,covers,allow_pickle=False)
    np.save(sp,stegos,allow_pickle=False)
    pd.DataFrame({'source_id':ids}).to_csv(ip,index=False)
    meta={
        'label':label,'payload_bpp':float(payload),'block_size':bs,
        'requested_sources':len(frame),'feasible_pairs':len(ids),
        'feasible_fraction':len(ids)/len(frame),'skipped':skipped,
        'source_ids_sha256':expected_hash,'protocol_sha256':protocol_sha,
    }
    mp.write_text(json.dumps(meta,indent=2),encoding='utf-8')
    del covers,stegos
    gc.collect()
    return np.load(cp,mmap_mode='r'),np.load(sp,mmap_mode='r'),ids

payloads=[0.050,0.030,0.015,0.009]
training_by_payload={}
validation_by_payload={}
cache_rows=[]
for p in payloads:
    fc,fs,fi=load_or_build_pairs(srnet_fit,'fit',p)
    dc,ds,di=load_or_build_pairs(srnet_dev,'dev',p)
    training_by_payload[float(p)]=(fc,fs)
    validation_by_payload[float(p)]=(dc,ds)
    cache_rows.append({
        'payload_bpp':p,
        'fit_feasible_pairs':len(fi),'fit_feasible_fraction':len(fi)/FIT_N,
        'dev_feasible_pairs':len(di),'dev_feasible_fraction':len(di)/DEV_N,
    })

cache_summary=pd.DataFrame(cache_rows)
cache_summary.to_csv(OUT/'srnet_v13_pair_feasibility.csv',index=False)
display(cache_summary)

if cache_summary.loc[np.isclose(cache_summary.payload_bpp,0.050),'fit_feasible_pairs'].iloc[0] < 1500:
    raise RuntimeError('Too few feasible 0.050-bpp fit pairs for the planned bootstrap.')
if cache_summary.loc[np.isclose(cache_summary.payload_bpp,0.050),'dev_feasible_pairs'].iloc[0] < 200:
    raise RuntimeError('Too few feasible 0.050-bpp development pairs for a stable progress gate.')

## 4. Progressive SRNet training

The best development-AUC checkpoint from each payload seeds the next lower payload. On CPU, the first decision point is the 0.050-bpp progress gate after epoch 4. If the best AUC is still below 0.55, the helper halts automatically.

In [ ]:
srnet,history,stage_summaries,device,run_status=train_progressive_srnet_v13(
    stage_specs=STAGES,
    training_by_payload=training_by_payload,
    validation_by_payload=validation_by_payload,
    pair_batch_size=PAIR_BATCH,
    weight_decay=WEIGHT_DECAY,
    seed=seed+13000,
    device=None,
    fixed_fpr=fixed_fpr,
    checkpoint_dir=CKPT,
    protocol_sha256=protocol_sha,
    resume=True,
    log_interval=100,
)

hist=pd.DataFrame(history)
hist.to_csv(OUT/'srnet_v13_train_history.csv',index=False)
stages_df=pd.DataFrame(stage_summaries)
stages_df.to_csv(OUT/'srnet_v13_stage_summary.csv',index=False)

display(hist)
display(stages_df)
print('Run status:',json.dumps(run_status,indent=2))
print('Device:',device)
print('Parameter count:',parameter_count(srnet))

if run_status['status']!='COMPLETE':
    failure={
        **run_status,
        'protocol_sha256':protocol_sha,
        'test_split_scored':False,
        'allocator_retuned':False,
    }
    (OUT/'srnet_v13_failed_progress_gate.json').write_text(json.dumps(failure,indent=2),encoding='utf-8')
    raise RuntimeError(
        f"SRNet-v13 halted at {run_status.get('failed_stage')} with best DEV AUC "
        f"{run_status.get('best_dev_auc'):.4f}. TEST was not scored."
    )

## 5. Final 0.009-bpp development sanity gate

This is the only gate that can authorize a **future, separate** TEST-scoring patch. Passing it does not itself score TEST.

In [ ]:
TARGET_BPP=0.009
dev_c,dev_s=validation_by_payload[TARGET_BPP]
dev_cover_scores=score_srnet(srnet,dev_c,device=device,batch_size=max(4,PAIR_BATCH))
dev_stego_scores=score_srnet(srnet,dev_s,device=device,batch_size=max(4,PAIR_BATCH))

y=np.tile([0,1],len(dev_cover_scores))
sc=np.column_stack([dev_cover_scores,dev_stego_scores]).reshape(-1)
met=detector_metrics(y,sc,fixed_fpr)
from sklearn.metrics import roc_curve
fpr_arr,tpr_arr,_=roc_curve(y,sc)
dev_pe=float(np.min(0.5*(fpr_arr+(1.0-tpr_arr))))
ci=paired_detector_bootstrap(
    dev_cover_scores,dev_stego_scores,fixed_fpr=fixed_fpr,
    n_resamples=max(n_boot,3000),confidence=confidence,seed=seed+13100,
)

selected_ckpt=Path(run_status['selected_checkpoint'])
model_path=MODEL_DIR/'srnet_secondary_v13.pt'
save_state_dict(srnet,model_path)

sanity={
    'sanity_pass':bool(met['auc']>=SANITY_AUC_THRESHOLD),
    'sanity_auc_threshold':SANITY_AUC_THRESHOLD,
    'pairs':len(dev_cover_scores),
    'auc':float(met['auc']),
    'auc_ci_low':float(ci['auc_low']),'auc_ci_high':float(ci['auc_high']),
    'tpr_at_5pct_fpr':float(met['tpr_at_fpr']),
    'tpr_ci_low':float(ci['tpr_low']),'tpr_ci_high':float(ci['tpr_high']),
    'pe':dev_pe,
    'selected_checkpoint':str(selected_ckpt),
    'selected_checkpoint_sha256':sha256_file(selected_ckpt),
    'model_sha256':sha256_file(model_path),
    'protocol_sha256':protocol_sha,
    'test_split_scored':False,
    'allocator_retuned':False,
}
(OUT/'srnet_v13_dev_sanity.json').write_text(json.dumps(sanity,indent=2),encoding='utf-8')
print(json.dumps(sanity,indent=2))

fig,ax=plt.subplots(figsize=(7.0,4.2))
for stage,g in hist.groupby('stage',sort=False):
    ax.plot(g.global_epoch,g.dev_auc,marker='o',label=stage)
ax.axhline(0.5,linewidth=1)
ax.axhline(SANITY_AUC_THRESHOLD,linestyle='--',linewidth=1)
ax.set_xlabel('Global training epoch')
ax.set_ylabel('Development ROC-AUC')
ax.set_title('SRNet-v13 progressive curriculum')
ax.legend(frameon=False,fontsize=8)
fig.tight_layout()
fig.savefig(OUT/'srnet_v13_dev_auc_history.png',dpi=300)
fig.savefig(OUT/'srnet_v13_dev_auc_history.svg')
plt.show()

In [ ]:
lock={
    **protocol,
    'protocol_sha256':protocol_sha,
    'srnet_v13_model_sha256':sha256_file(model_path),
    'dev_sanity_auc':sanity['auc'],
    'dev_sanity_pass':sanity['sanity_pass'],
    'selected_checkpoint_sha256':sanity['selected_checkpoint_sha256'],
    'protocol_locked_before_any_srnet_v13_test_scoring':True,
    'test_split_scored':False,
}
lock_path=OUT/'srnet_v13_pretest_lock.json'
lock_path.write_text(json.dumps(lock,indent=2),encoding='utf-8')
print('PRETEST LOCK SHA256:',sha256_file(lock_path))

if not sanity['sanity_pass']:
    raise RuntimeError(
        f"SRNet-v13 failed the final pre-test gate: AUC={sanity['auc']:.3f} < {SANITY_AUC_THRESHOLD:.2f}. "
        'TEST was not scored. Preserve this detector-development attempt.'
    )

print('FINAL DEV SANITY PASSED.')
print('STOP HERE. Patch 13 contains no TEST scoring.')
print('Preserve srnet_v13_pretest_lock.json and request a separate Patch 14 for frozen TEST scoring.')

## Stop condition

Patch 13 intentionally ends here. Even if the final development gate passes, **do not add TEST-scoring code to this notebook**. Preserve the model and the pre-test lock, then create a separate frozen TEST-scoring patch.